<a href="https://colab.research.google.com/github/vedantpople4/LLM/blob/main/FT_BERT_MovieRecommendation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install datasets evaluate transformers[sentencepiece]
!pip install accelerate -U
!pip install peft

In [ ]:
from datasets import load_dataset, DatasetDict, Dataset

from transformers import (
    AutoTokenizer,
    AutoConfig,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer)

from peft import PeftModel, PeftConfig, get_peft_model, LoraConfig
import evaluate
import torch
import numpy as np

In [ ]:
imdb_dataset = load_dataset("imdb")

N = 500

rand_idx = np.random.randint(24999, size=N)

x_train = imdb_dataset['train'][rand_idx]['text']
y_train = imdb_dataset['train'][rand_idx]['label']

x_test = imdb_dataset['test'][rand_idx]['text']
y_test = imdb_dataset['test'][rand_idx]['label']

dataset = DatasetDict({'train':Dataset.from_dict({'label':y_train,'text':x_train}),
                             'validation':Dataset.from_dict({'label':y_test,'text':x_test})})

In [ ]:
#dataset['train'][0]

In [ ]:
np.array(dataset['train']['label']).sum()/len(dataset['train']['label'])

In [ ]:
model_checkpoint = 'distilbert-base-uncased'
# model_checkpoint = 'roberta-base'

id2label = {0: "Negative", 1: "Positive"}
label2id = {"Negative":0, "Positive":1}

model = AutoModelForSequenceClassification.from_pretrained(
    model_checkpoint, num_labels=2, id2label=id2label, label2id=label2id)

In [ ]:
#model

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint, add_prefix_space=True)

if tokenizer.pad_token is None:
    tokenizer.add_special_tokens({'pad_token': '[PAD]'})
    model.resize_token_embeddings(len(tokenizer))

In [ ]:
def tokenize_function(examples):
    text = examples["text"]
    tokenizer.truncation_side = "left"
    tokenized_inputs = tokenizer(
        text,
        return_tensors="np",
        truncation=True,
        max_length=512
    )

    return tokenized_inputs

In [ ]:
tokenized_dataset = dataset.map(tokenize_function, batched=True)
tokenized_dataset

In [ ]:
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [ ]:
accuracy = evaluate.load("accuracy")

In [ ]:
def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=1)

    return {"accuracy": accuracy.compute(predictions=predictions, references=labels)}

In [ ]:
sentiment_examples = [
    "The movie was an absolute delight from start to finish!",
    "I wouldn't recommend this film to anyone; it was quite a letdown.",
    "A truly captivating experience with stunning performances and a gripping plot.",
    "This film is a waste of time; it fails to engage or entertain.",
    "One of the best movies I've seen this year; it exceeded all my expectations.",
    "Unfortunately, this movie did not live up to the hype; it was very disappointing.",
    "A feel-good movie with a perfect balance of humor and drama.",
    "The plot was predictable and the characters were bland. Not worth watching.",
    "This movie is a must-watch; it?s both entertaining and thought-provoking.",
    "I was bored throughout; the story was lackluster and the pacing was off."
]


print("Untrained model predictions:")
print("----------------------------")
for text in sentiment_examples:
    inputs = tokenizer.encode(text, return_tensors="pt")
    logits = model(inputs).logits
    predictions = torch.argmax(logits)
    print(text + " - " + id2label[predictions.tolist()])

In [ ]:
sentiment_examples_with_category = {
    "The movie was an absolute delight from start to finish!": "Positive",
    "I wouldn't recommend this film to anyone; it was quite a letdown.": "Negative",
    "A truly captivating experience with stunning performances and a gripping plot.": "Positive",
    "This film is a waste of time; it fails to engage or entertain.": "Negative",
    "One of the best movies I've seen this year; it exceeded all my expectations.": "Positive",
    "Unfortunately, this movie did not live up to the hype; it was very disappointing.": "Negative",
    "A feel-good movie with a perfect balance of humor and drama.": "Positive",
    "The plot was predictable and the characters were bland. Not worth watching.": "Negative",
    "This movie is a must-watch; it?s both entertaining and thought-provoking.": "Positive",
    "I was bored throughout; the story was lackluster and the pacing was off.": "Negative"
}


In [ ]:
peft_config = LoraConfig(task_type="SEQ_CLS",
                        r=4,
                        lora_alpha=32,
                        lora_dropout=0.01,
                        target_modules = ['q_lin'])

peft_config

In [ ]:
model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

In [ ]:
lr = 1e-3
batch_size = 4
num_epochs = 1

In [ ]:
training_args = TrainingArguments(
    output_dir= model_checkpoint + "-lora-text-classification",
    learning_rate=lr,
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    num_train_epochs=num_epochs,
    weight_decay=0.01,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
)

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

In [ ]:
trainer.train()

In [ ]:
print("Trained model predictions:")
print("--------------------------")
for text in sentiment_examples:
    inputs = tokenizer.encode(text, return_tensors="pt")
    inputs = inputs.to(model.device)
    logits = model(inputs).logits
    predictions = torch.max(logits,1).indices
    print(text + " - " + id2label[predictions.tolist()[0]])